# 02. Naive Price Elasticity & Confounding Analysis

Compare Linear, Naive Log-Log, Controlled Log-Log, and Segment-level regressions, and demonstrate Simpson's Paradox.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

from simulation.data_generator import generate_synthetic_marketplace_data
from preprocessing.cleaning import clean_transaction_data
from preprocessing.feature_engineering import add_engineered_features
from analysis.elasticity import run_elasticity_comparison
from analysis.confounding import analyze_correlations_and_confounders, demonstrate_simpsons_paradox

feat_df = add_engineered_features(clean_transaction_data(generate_synthetic_marketplace_data(n_rows=15000, seed=42))[0])
el_res = run_elasticity_comparison(feat_df)
el_res['summary_table'][['model_name', 'elasticity', 'std_error', 'r_squared', 'true_avg_elasticity', 'bias_vs_true']]

2026-11-27 05:26:00 | INFO     | preprocessing.cleaning | Cleaned dataset: 15000 -> 15000 rows (dups=0, invalid=0, imputed=0, clipped=45)


,model_name,elasticity,std_error,r_squared,true_avg_elasticity,bias_vs_true
0,Linear OLS (Mean Elasticity),0.166343,0.009871,0.028912,-1.019471,1.185814
1,Naive Log-Log OLS,0.093745,0.007753,0.009485,-1.019471,1.113216
2,Controlled Log-Log OLS,-0.582891,0.011091,0.678807,-1.019471,0.436580


In [2]:
el_res['segment_controlled'].query("segment_type == 'region'")

,segment_type,segment_value,elasticity,std_error,ci_low,ci_high,p_value,r_squared,n_obs,true_elasticity,bias
0,region,East,-0.767047,0.021723,-0.809623,-0.724471,3.996052e-273,0.447852,3422,-1.224531,0.457484
1,region,North,-0.923229,0.018335,-0.959165,-0.887292,0.000000e+00,0.494597,4335,-1.418047,0.494819
2,region,South,-0.411249,0.020821,-0.452058,-0.370440,7.807074e-87,0.315052,3775,-0.833371,0.422121
3,region,West,-0.097075,0.021269,-0.138761,-0.055389,5.013878e-06,0.263898,3468,-0.521487,0.424413


In [3]:
simpson = demonstrate_simpsons_paradox(feat_df, group_col='region')
print(simpson['explanation'])
simpson['within_segment_elasticities']

Pooled naive log-log slope is +0.094, whereas within-region slopes (accounting for product reference/category mix) range from -0.923 to -0.097. Aggregating across heterogeneous regions confounds price sensitivity with baseline demand levels.


,segment_type,segment_value,elasticity,std_error,ci_low,ci_high,p_value,r_squared,n_obs,true_elasticity,bias
0,region,East,-0.767047,0.021723,-0.809623,-0.724471,3.996052e-273,0.447852,3422,-1.224531,0.457484
1,region,North,-0.923229,0.018335,-0.959165,-0.887292,0.000000e+00,0.494597,4335,-1.418047,0.494819
2,region,South,-0.411249,0.020821,-0.452058,-0.370440,7.807074e-87,0.315052,3775,-0.833371,0.422121
3,region,West,-0.097075,0.021269,-0.138761,-0.055389,5.013878e-06,0.263898,3468,-0.521487,0.424413


In [4]:
conf = analyze_correlations_and_confounders(feat_df)
conf['confounder_screen'].head(8)

,variable,corr_with_log_price,corr_with_log_quantity,abs_product,role_note
0,log_competitor_price,0.887989,0.223983,0.198894,Potential confounder / covariate (requires dom...
1,supplier_cost_shock,0.393341,-0.399591,0.157176,Excluded cost instrument candidate (correlates...
2,discount,-0.271091,-0.136741,0.037069,Potential confounder / covariate (requires dom...
3,promotion_flag,-0.159986,0.048280,0.007724,Potential confounder / covariate (requires dom...
4,seller_availability,0.032916,0.159428,0.005248,Potential confounder / covariate (requires dom...
5,month,-0.054700,0.029006,0.001587,Potential confounder / covariate (requires dom...
6,festival_flag,0.017329,0.066099,0.001145,Potential confounder / covariate (requires dom...
7,log_historical_orders,-0.003744,0.152184,0.000570,Potential confounder / covariate (requires dom...
